# Scikit-Fuzzy — Laboratorio de Sistemas Difusos
### IPD434 — Computación Evolutiva e Inteligencia Artificial
**Universidad Técnica Federico Santa María**

---

## 🎯 Objetivos de Aprendizaje

Al finalizar esta clase, el estudiante será capaz de:

1. **Implementar** funciones de membresía con `scikit-fuzzy` (`fuzz.trimf`, `fuzz.gaussmf`, …).
2. **Construir** un sistema de control difuso con `ctrl.Antecedent`, `ctrl.Consequent` y `ctrl.ControlSystem`.
3. **Definir** el conjunto de reglas difusas y simular el sistema con `ctrl.ControlSystemSimulation`.
4. **Analizar** la superficie de control y la sensibilidad del sistema al diseño de las reglas.
5. **Conectar** la teoría del notebook `02_SistemasDifusos` con su implementación práctica.

> 🧪 **Laboratorio:** este cuaderno es la contraparte práctica del módulo de Sistemas Difusos.

> 💡 **Prerrequisitos:** [02_SistemasDifusos](02_SistemasDifusos.ipynb).

## ¿Qué es `scikit-fuzzy`?

**`scikit-fuzzy`** (`skfuzzy`) es una biblioteca de Python para construir **sistemas de inferencia difusa (FIS)**, desarrollada sobre el stack científico de **SciPy/NumPy**. Nos entrega:

- Funciones de **membresía** listas para usar (`fuzz.trimf`, `fuzz.trapmf`, `fuzz.gaussmf`, `fuzz.pimf`, `fuzz.smf`, `fuzz.zmf`, …).
- Un módulo de **control** (`skfuzzy.control`) para declarar variables lingüísticas (`ctrl.Antecedent`, `ctrl.Consequent`), reglas `SI … ENTONCES` (`ctrl.Rule`) y ensamblarlas en un `ctrl.ControlSystem`.
- Un **simulador** (`ctrl.ControlSystemSimulation`) que *fuzzifica* las entradas, aplica las reglas y *defuzzifica* la salida a un número concreto.

> Un **sistema de inferencia difusa (FIS)** traduce entradas numéricas a etiquetas lingüísticas (p. ej. *"la comida es piola"*), razona con reglas expresadas en lenguaje natural y devuelve una decisión numérica.

## Clase práctica: del concepto a la implementación

En el cuaderno teórico ([02_SistemasDifusos](02_SistemasDifusos.ipynb)) vimos *qué* son los conjuntos difusos y *cómo* razona un FIS. Aquí lo llevamos a código: partiremos de un caso cotidiano —**cuánta propina dejar**— para recorrer, paso a paso, el flujo completo de un controlador difuso con `scikit-fuzzy`.

## Ejemplo 1: Propina

Un problema clásico de lógica difusa: decidir la propina combinando dos percepciones subjetivas (calidad de la comida y atención del mesero). Es ideal porque las entradas son *vagas* ("estuvo piola") y, aun así, queremos una salida numérica bien definida.

Ud. acude al restaurant **El maestro difuso**, el rey de la comida universitaria cercano a la **Universidad Difusa Federico Santa María**. Ud. como buen comensal, desea dejar propina, pero su decisión estará basada en la calidad de la comida y la atención del mesero: Si las cosas van mal, no dejará propina y si van de forma inigualable dejará hasta 15%, vale decir, añadirá hasta un 5% extra al 10% sugerido. El resto variará entre una fracción del 10% base y el sugerido normalmente.

Ahora, considerando el común nacional, las evaluaciones se hacen en una escala del 1 al 7 con décimas. Sin embargo, ud. define sus escalas subjetivas de las siguientes formas ascendentes:
* Comida: `pesimo`, `malito`, `piola`, `de pana`, `basadísimo`.
* Atención: `amarga`, `pesada`, `noesni`, `tela`, `joya`.
Además, no tiene claro que valor de la escala representa cada una de sus calificaciones. 

**Construya un sistema de lógica difusa que le permita resolver cuánta propina debiese dejar según la probable atención en el restaurant**. Defina claramente variables numéricas, difusas, funciones de pertenencia, reglas difusas. Además explique, ¿Cuándo ocurre la Fuzzificación y la Defuzzificación?

<center><img src="images/fuzzylogic_system.png" align="center"/></center>

<center><em>Arquitectura de un sistema de inferencia difusa: fuzzificación → motor de reglas → defuzzificación.</em></center>

<center><img src="images/fuzzy_steps.png" align="center"/></center>

<center><em>Etapas del razonamiento difuso, paso a paso.</em></center>

### Variables del sistema

Todo FIS se define sobre **variables lingüísticas**: variables cuyos valores son etiquetas (*"pésima"*, *"joya"*) en lugar de números. Cada variable vive en un **universo de discurso** (el rango numérico donde está definida). Distinguimos **antecedentes** (entradas) y **consecuentes** (salidas).

#### Variables de entrada (antecedentes)
* ¿Qué tan bueno estuvo el almuerzo?:
    * Variable: `comida`.
    * Como variable numérica, universo: $\{1.0, 1.1, 1.2, 1.3,\dots,6.7, 6.8, 6.9, 7.0\}$
    * Como variable difusa: $\{\texttt{pesimo}, \texttt{ malito}, \texttt{ piola}, \texttt{ de pana}, \texttt{ basadísimo}\}$

* ¿Qué tan bien te atendieron?:
    * Variable: `atención`.
    * Como variable numérica, universo: $\{1.0, 1.1, 1.2, 1.3,\dots,6.7, 6.8, 6.9, 7.0\}$
    * Como variable difusa: $\{\texttt{amarga}, \texttt{ pesada}, \texttt{ noesni}, \texttt{ tela}, \texttt{ joya}\}$

#### Variables de salida (consecuente)
* ¿Cuánta propina voy a dejar?
    * Variable: `propina`
    * Como variable numérica, universo (en %): $\{0, 1, 2, 3, \dots, 9, 10, 11, 12, 13, 14, 15\}$
    * Como variable difusa: $\{\texttt{nada}, \texttt{ poco}, \texttt{ normal}, \texttt{ extra}\}$

In [ ]:
# Incluir librerías
import numpy as np
import skfuzzy as fuzz

from skfuzzy import control as ctrl

In [ ]:
# Variable de Entradas o Antecedentes
calidad = ctrl.Antecedent(np.arange(1, 7.1, 0.1), 'calidad') #pésima, malita, piola, de pana, basadísimo
atención = ctrl.Antecedent(np.arange(1, 7.1, 0.1), 'atención') #amarga, pesada, noesni, tela, joya

#Variable de salida o Consecuentes
propina = ctrl.Consequent(np.arange(0, 15.1, 0.1), 'propina')

#.universe es un método para Antecedentes y Consecuentes, que entrega el dominio de la variable numérica
print(propina.universe) 

### Funciones de membresía

Una **función de membresía** $\mu_A(x) \in [0,1]$ indica *cuánto* pertenece un valor $x$ a un conjunto difuso $A$. A diferencia de la lógica clásica (pertenece / no pertenece), aquí la pertenencia es **gradual**. `scikit-fuzzy` ofrece varias formas:

| Función | Forma | Uso típico |
|---------|-------|-----------|
| `fuzz.trimf` | triangular | conjunto centrado en un valor |
| `fuzz.trapmf` | trapezoidal | conjunto con "meseta" plana |
| `fuzz.gaussmf` | gaussiana | transiciones suaves |
| `fuzz.pimf` | tipo $\pi$ | meseta con hombros curvos |
| `fuzz.smf` / `fuzz.zmf` | sigmoides S / Z | umbrales "hacia arriba" / "hacia abajo" |

A continuación asignamos una función de membresía a **cada etiqueta** de cada variable.

In [ ]:
# Definir funciones de membresía
# Cada valor de la variable difusa debe tener una función de membresía
# funciones de referencia en: https://pythonhosted.org/scikit-fuzzy/api/skfuzzy.membership.html
calidad['pésima'] = fuzz.gaussmf(calidad.universe, 1,0.5)
calidad['malita'] = fuzz.trimf(calidad.universe,[1.5,3,4.5])
calidad['piola'] = fuzz.pimf(calidad.universe, 3,4,5,6)
calidad['de pana'] = fuzz.trapmf(calidad.universe,[5,5.5,6.0,6.5])
calidad['basadísimo'] = fuzz.smf(calidad.universe,6.0,7)

#graficar membresías específica utilizando el operador ['categoria'], i.e., calidad['pesimo'].view()
calidad.view()
calidad["pésima"].view()

In [ ]:
# Existe un método de generación automática de función de membresía triangular
# Funciona solo con los valores 3, 5, o 7.
# Para otra cantidad de categorías, se debe hacer manualmente o pasar una lista con n labels 
nombres = ["amarga", "pesada", "noesni", "tela", "joya"]
atención.automf(names=nombres)
atención.view()

In [ ]:
# Consecuente y funciones de pertenencia
salida=["nada", "poca", "normal", "extra"]
propina.automf(names=salida) #comenzamos con automático
propina["nada"]=fuzz.zmf(propina.universe,0,1) #sobreescribimos algunas
propina["poca"]=fuzz.gaussmf(propina.universe, 5, 1.25)
propina["extra"]=fuzz.smf(propina.universe,10,15)
propina.view()

### Reglas difusas

El **conocimiento experto** se codifica como reglas `SI <antecedente> ENTONCES <consecuente>`. En `scikit-fuzzy` cada regla es un `ctrl.Rule`, y combinamos antecedentes con los operadores difusos **`&` (AND / mínimo)** y **`|` (OR / máximo)**.

```Si las cosas van mal, no dejará propina y si van de forma inigualable dejará hasta 15%, vale decir, añadirá hasta un 5% extra al 10% sugerido. El resto variará entre una fracción del 10% base y el sugerido normalmente.```

* **Sí** la calidad de la comida es _pésima_ y la atención del mesero es _amarga_, **entonces** la propina será _nada_.
* **Sí** la calidad de la comida es _malita_ o _mas o menos_, **entonces** la propina será _poca_.
* **Sí** la atención del mesero es _pesada_ o _noesni_, **entonces** la propina será _poca_.
* **Sí** la calidad de la comida es _de pana_, **entonces** la propina será _normal_.
* **Sí** la atención del mesero es _tela_, **entonces** la propina será _normal_.
* **Sí** la calidad de la comida es _basadísima_ y la atención del mesero es _joya_, **entonces** la propina será _extra_.


In [ ]:
# Cada regla relaciona antecedentes con consecuentes. Los resuelve segun su valor de la función de membresía.
regla1 = ctrl.Rule(calidad['pésima'] & atención['amarga'], propina['nada'])
regla2 = ctrl.Rule(calidad['malita'] | calidad['piola'], propina['poca'])
regla3 = ctrl.Rule(atención['pesada'] | atención['noesni'], propina['poca'])
regla4_5 = ctrl.Rule((calidad['de pana'] | atención['tela']), propina['normal'])
regla6 = ctrl.Rule(calidad['basadísimo'] & atención['joya'], propina['extra'])

#pésima, malita, piola, de pana, basadísimo
# View permite ver la complejidad del tema, donde los nodos de termino son los posibles valores que puede tomar el consecuente.
# regla6.view() no funciona en esta versión(?)

### Sistema de control, simulación y (de)fuzzificación

Con las variables, las membresías y las reglas ya definidas, ensamblamos todo en un `ctrl.ControlSystem` y creamos una `ctrl.ControlSystemSimulation` para evaluarlo con entradas concretas.

**¿Cuándo ocurren la fuzzificación y la defuzzificación?**

1. **Fuzzificación:** al hacer `sim.input[...] = valor` y luego `sim.compute()`, cada entrada numérica se convierte en grados de pertenencia a las etiquetas (p. ej. `4.32` → *"malita"* con grado alto y *"piola"* con grado menor).
2. **Inferencia:** se activan las reglas y se agregan sus consecuentes en un conjunto difuso de salida.
3. **Defuzzificación:** ese conjunto se colapsa a un único número (por defecto, el **centroide**), que leemos en `sim.output[...]`.

In [ ]:
#Creamos un Sistema de Control Difuso, el cual combina todas las reglas creadas.
control_propina = ctrl.ControlSystem([regla1, regla2, regla3, regla4_5, regla6])

In [ ]:
#Creamos una Simulación del Sistema de Control Difuso. 
dejar_propina = ctrl.ControlSystemSimulation(control_propina)

#A esta simulación, le podemos asignar, según nombre de los antecedentes, un valor de entrada númerico.
dejar_propina.input['calidad'] = 4.32
dejar_propina.input['atención'] = 5.123

# Luego, podemos computar según el proceso el Sistema Difuso creado
dejar_propina.compute()


In [ ]:
#Podemos imprimir la salida numérica (desfuzzificada)

print("\nLa propina debe ser de un", str(dejar_propina.output['propina'])+"%\n")


In [ ]:
#Podemos ver los gráficos de las funciones de membresía sobre los valores de los antecedentes y consecuentes. 
calidad.view(sim=dejar_propina)
atención.view(sim=dejar_propina)
propina.view(sim=dejar_propina)

## Ejercicio 1: Aire acondicionado

Aplicamos ahora todo lo anterior a un caso de control más rico: un aire acondicionado en modo **auto** que decide, por sí solo, el **modo** (frío / stand-by / calor) y la **velocidad** del ventilador a partir de la temperatura ambiente y su diferencia con la temperatura deseada.

Ud. ha sido contratado por IPD-Electrónics para trabajar en su marca TEL-Air que comercializará equipos de aire acondicionado. Estos dispositivos climatizan ambientes interiores en el rango entre $17ºC$ a $30ºC$, lanzando aire frio o aire caliente dependiendo de las instrucciones que el usuario le entregue, hasta llegar a la temperatura deseada. Así:

* Para climatizar utilizando el modo _calor_, el aire acondicionado lanza aire caliente constante, a unos $42ºC$, hasta llegar a la temperatura deseada. 
* Para climatizar utilizando el modo _frío_, el aire acondicionado lanza aire fresco constante, a unos $13ºC$, hasta llegar a la temperatura deseada.
* Para mantener el clima actual del ambiente se utiliza el modo _stand by_, que provee de circulación del aire sin modificar la variación normal de la temperatura.
* Para climatizar automáticamente se utiliza el modo _auto_, el cual se encarga de escoger el modo del aire acondicionado, y el nivel de velocidad utilizado.

El aire es lanzado por los dispositivos en tres posibles velocidades: baja, media o alta. Estas velocidades modifican la temperatura, según el modo de aire y las características de la zona climatizada, en los rangos (en $\frac{ºC}{\text{min}}$) $[0.1, 0.7]$, $[0.5, 1.2]$ y $[0.8, 1.9]$, respectivamente. Considere que si está en modo _stand by_ estas variaciones se darán para acercarse a la temperatura del exterior.  

Para controlar la temperatura del ambiente, los dispositivos cuentan con un termostato que les indica la temperatura actual de la zona climatizada.

Considere que para limitar el consumo energético, el modo _auto_ define las siguientes reglas bases:
* Si la diferencia de temperatura es negativa, debe utilizar modo calor.
* Si la diferencia de temperatura es positiva, debe utilizar modo frío.
* Si no hay diferencia de temperatura, debe utilizar el modo stand by.
* Si se encuentra cercano o en la temperatura deseada, no puede utilizar el modo de velocidad alta.
* Si se encuentra muy alejado en la temperatura, debe utilizar el modo de velocidad alta. 

Y que además, utilizando el historico de temperatura actual y de destino, reflejado en las configuraciones manuales, se ha determinado qué:
* La temperatura puede ser catalogada como fría si está bajo el umbral de los 15ºC.
* La temperatura puede ser catalogada como fresca si está en el rango de los 12.5ºC a los 20ºC.
* La temperatura puede ser catalogada como ambiente entre los 18ºC a los 25ºC.
* La temperatura puede ser catalogada como calurosa sobre el rango de los 23ºC.

Y como es un sistema basado en la percepción y el comfort humano, se definen las siguientes reglas básicas:
* Si la temperatura es fría, el aire debe utilizar modo calor. 
* Si la temperatura es calurosa, el aire debe utilizar modo frío.

1. Genere un sistema de inferencia difusa que permita para un instante determinado, utilizando el modo _auto_, determinar el modo de función del dispositivo y la velocidad con la cual lanza el aire. Determine claramente los antecendentes y los consecuentes del sistema, funciones de membresía y las reglas de inferencia difusa del sistema.

In [ ]:
# Incluir librerías
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

### Variables

Definimos los **antecedentes** (entradas) y **consecuentes** (salidas) del controlador, cada uno con su universo de discurso.

#### Antecedentes
- **`ambiente`**: temperatura actual de la zona, $0$–$40\,°C$ (etiquetas: *frío, fresco, ambiente, caluroso*).
- **`diferencia`**: temperatura actual menos la deseada (etiquetas: *muy negativo, negativo, neutro, positivo, muy positivo*).

In [ ]:
# Antecedentes: temperatura ambiente y diferencia respecto a la deseada
amb = ctrl.Antecedent(np.arange(0, 40.5, 0.5), 'ambiente') #frio, fresco, ambiente, caluroso
dif = ctrl.Antecedent(np.arange(-30, 23.5, 0.5), 'diferencia') #muy negativo, negativo, neutro, positivo, muy positivo

#### Consecuentes
- **`velocidad`** del ventilador: *baja, media, alta*.
- **`modo`** de operación, codificado en $[-1, 1]$: *frío ($-1$), stand-by ($0$), calor ($+1$)*.

In [ ]:
# Consecuentes: velocidad del ventilador y modo de operación
vel = ctrl.Consequent(np.arange(0.1, 2.0, 0.1), "velocidad") # baja, media, alta
modo = ctrl.Consequent(np.arange(-1,1.01, 0.01), "modo") #frio, stand by, calor

### Funciones de membresía

Asignamos a cada etiqueta su función de membresía. Usamos `pimf` (trapecios de hombros suaves) para las velocidades y `gaussmf`/`smf`/`zmf` para modelar el centro y los extremos del `modo` y de la temperatura.

In [ ]:
# Membresías de la temperatura ambiente
amb['frio'] = fuzz.zmf(amb.universe, 0,15)
amb['fresco'] = fuzz.pimf(amb.universe, 12.5, 15, 18, 20)
amb['ambiente'] = fuzz.gaussmf(amb.universe, 21, 1.5)
amb['caluroso'] = fuzz.smf(amb.universe, 23, 40)
amb.view()

In [ ]:
# Membresías de la diferencia de temperatura (generadas con automf)
diffs = ['muy negativo', 'negativo', 'neutro', 'positivo', 'muy positivo']
dif.automf(names=diffs)
dif.view()

In [ ]:
# Membresías de la velocidad del ventilador
vel['baja'] = fuzz.pimf(vel.universe, 0.0, 0.15, 0.65, 0.8)
vel['media'] = fuzz.pimf(vel.universe, 0.3, 0.45, 1.15, 1.3)
vel['alta'] = fuzz.pimf(vel.universe, 0.7, 0.85, 1.85, 2.0)
vel.view()

In [ ]:
# Membresías del modo de operación
modo['frio'] = fuzz.zmf(modo.universe,-1,0)
modo['stand by'] = fuzz.gaussmf(modo.universe, 0, 0.1) 
modo['calor'] = fuzz.smf(modo.universe, 0, 1)
modo.view()

### Reglas difusas

Traducimos las reglas de negocio del enunciado a reglas `SI … ENTONCES`.

**Ahorro energético (según la diferencia de temperatura):**
* Si la diferencia es negativa → modo **calor**.
* Si la diferencia es positiva → modo **frío**.
* Si no hay diferencia → modo **stand-by**.
* Si estamos cerca de la temperatura deseada → **no** usar velocidad alta.
* Si estamos muy alejados de la temperatura → usar velocidad **alta**.

**Confort humano (según la temperatura ambiente):**
* Si la temperatura es fría → modo **calor**.
* Si la temperatura es calurosa → modo **frío**.

In [ ]:
# Reglas SI...ENTONCES y ensamblado del sistema de control
r1 = ctrl.Rule(dif['negativo'] | dif['muy negativo'], modo['calor'])
r2 = ctrl.Rule(dif['positivo'] | dif['muy positivo'], modo['frio'])
r3 = ctrl.Rule(dif['neutro'], modo['stand by'])
r4_1 = ctrl.Rule(dif['neutro'] | dif['negativo'] | dif['positivo'], vel['baja'])
r4_2 = ctrl.Rule(dif['neutro'] | dif['negativo'] | dif['positivo'], vel['media'])
r5 = ctrl.Rule(dif['muy negativo'] | dif['muy positivo'], vel['alta'])

r6 = ctrl.Rule(amb['frio'], modo['calor'])
r7 = ctrl.Rule(amb['caluroso'], modo['frio'])

ctrl_aire = ctrl.ControlSystem([r1, r2, r3, r4_1, r4_2, r5, r6, r7])

In [ ]:
# Simulación para un caso concreto (temperatura ambiente y destino dados)
AC = ctrl.ControlSystemSimulation(ctrl_aire)

t_ambiente = 20.0
t_destino = 17.0

AC.input['ambiente'] = t_ambiente
AC.input['diferencia'] = t_ambiente - t_destino

AC.compute()

In [ ]:
# Salida defuzzificada y visualización de las membresías activadas
print(AC.output['modo'])
print(AC.output['velocidad'])
dif.view(sim=AC)
amb.view(sim=AC)
modo.view(sim=AC)
vel.view(sim=AC)

2. Simule un sistema de control difuso que muestre cómo funcionaría el dispositivo durante **15 minutos**, actualizando las condiciones cada **30 segundos**.

> 🔬 **Ejercicio abierto (para el estudiante).** Con el controlador ya construido, escriba un bucle temporal que: (1) lea el `modo` y la `velocidad` en cada paso, (2) actualice la temperatura ambiente según la velocidad y el modo activos, y (3) recalcule la `diferencia`. Grafique la evolución de la temperatura hasta alcanzar (o aproximarse a) la deseada.

## 📌 Resumen

| Concepto | En `scikit-fuzzy` |
|----------|-------------------|
| Universo de discurso | `np.arange(inicio, fin, paso)` |
| Variable de entrada | `ctrl.Antecedent(universo, 'nombre')` |
| Variable de salida | `ctrl.Consequent(universo, 'nombre')` |
| Funciones de membresía | `fuzz.trimf`, `trapmf`, `gaussmf`, `pimf`, `smf`, `zmf` |
| Generación automática | `variable.automf(names=[...])` |
| Regla difusa | `ctrl.Rule(antecedente, consecuente)` (operadores AND `&`, OR) |
| Sistema de control | `ctrl.ControlSystem([reglas...])` |
| Simulación | `ctrl.ControlSystemSimulation(sistema)` + `.input` / `.compute()` / `.output` |
| Visualización | `variable.view()` y `variable.view(sim=...)` |

**Flujo de un FIS:** entrada numérica → **fuzzificación** → **inferencia** (reglas) → **agregación** → **defuzzificación** (centroide) → salida numérica.

## 🔗 Próximos pasos
- Completar el ejercicio abierto de simulación temporal del aire acondicionado.
- Revisar los cuadernos de desarrollo: [CCCV](02_SciKit_Fuzzy-Desarrollo-CCCV.ipynb) y [CSSJ](02_SciKit_Fuzzy-Desarrollo-CSSJ.ipynb).
- Módulo siguiente: [03_ComputacionEvolutiva](../03_ComputacionEvolutiva/03_ComputacionEvolutiva.ipynb).

## 📚 Referencias

- **scikit-fuzzy** — Documentación y API oficial: https://pythonhosted.org/scikit-fuzzy/ · Repositorio: https://github.com/scikit-fuzzy/scikit-fuzzy
- **Ejemplo *Tipping Problem*** (base del Ejemplo 1): https://pythonhosted.org/scikit-fuzzy/auto_examples/plot_tipping_problem_newapi.html
- **Zadeh, L. A. (1965).** *Fuzzy Sets*. Information and Control, 8(3), 338–353.
- **Mamdani, E. H., & Assilian, S. (1975).** *An experiment in linguistic synthesis with a fuzzy logic controller*. Int. J. Man-Machine Studies, 7(1), 1–13.
- **Ross, T. J. (2010).** *Fuzzy Logic with Engineering Applications* (3rd ed.). Wiley.